# Task 2: MIP Data Analysis (2016)

Analysis of MIP 2016 data following the assignment requirements.

## Packages Used:
- **pyfixest**: Fast high-dimensional fixed effects regression (fixest-style syntax)
- **marginaleffects**: Compute and plot marginal effects, predictions, and comparisons
- **statsmodels**: Statistical models and tests

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import pyfixest as pf
from marginaleffects import predictions, comparisons, slopes, avg_slopes, plot_predictions
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

# Plotting settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully')

INFO:visions.backends:Pandas backend loaded 2.3.3


INFO:visions.backends:Numpy backend loaded 2.3.5


INFO:visions.backends:Pyspark backend NOT loaded


INFO:visions.backends:Python backend loaded


Libraries loaded successfully


In [2]:
# Load data
df, meta = pyreadstat.read_dta('data/MIP 2016 data.dta')
print(f'Dataset: {df.shape[0]} observations, {df.shape[1]} variables')
print('\nVariable names:')
print(df.columns.tolist())

Dataset: 2000 observations, 96 variables

Variable names:
['id', 'ost', 'branche', 'bran_4', 'filter', 'bges', 'gk3n', 'bhsp', 'bhspno', 'um', 'lp', 'lpx', 'ex', 'exno', 'umpr1', 'pd', 'umneu', 'umunw', 'mneu', 'mneup', 'novor', 'novorp', 'pz', 'rek', 'rekp', 'qual', 'qualp', 'pdn', 'pzn', 'pxn', 'pda', 'pza', 'pxa', 'pdnano', 'pznano', 'pd2016', 'pz2016', 'px2016', 'pdzu2016', 'pdzn2016', 'pd2017', 'pz2017', 'px2017', 'pdzu2017', 'pdzn2017', 'iages', 'iainv', 'iainvno', 'iae2016', 'iae2017', 'iano2016', 'iano2017', 'iap2016', 'iap2016x', 'iap2017', 'iap2017x', 'fuekon', 'fueb', 'fueexti', 'fueexta', 'fueextno', 'fue', 'fueno', 'digia1', 'digia2', 'digia3', 'digia4', 'digif1', 'digif2', 'digif3', 'digif4', 'digia5', 'digia6', 'digia7', 'digif5', 'digif6', 'digif7', 'digia8', 'digia9', 'digif8', 'digif9', 'digia10', 'digia11', 'digif10', 'digif11', 'digisw1', 'digisw2', 'digisw3', 'digisw4', 'digisw5', 'digisw6', 'digisw7', 'digisw8', 'digisw9', 'digisw10', 'digisw11']


## 2. Descriptive Statistics

In [3]:
# Data exploration with ydata-profiling
profile = ProfileReport(df, title='MIP 2016 Data Profile', explorative=True)
profile.to_file('data/mip_2016_profile.html')
print('Profile report saved: data/mip_2016_profile.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/96 [00:00<?, ?it/s]

 85%|████████▌ | 82/96 [00:00<00:00, 508.39it/s]

100%|██████████| 96/96 [00:00<00:00, 487.68it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile report saved: data/mip_2016_profile.html


In [4]:
# Descriptive statistics for numerical variables
df.describe()

,id,ost,branche,bran_4,filter,bges,gk3n,bhsp,um,lp,...,rekp,qualp,iages,iainv,iap2016,iap2016x,iap2017,iap2017x,fueb,fue
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.0000,2000.000000,1990.000000,1856.000000,2000.000000,1990.000000,...,1332.000000,1309.000000,1773.000000,1732.000000,643.000000,643.000000,602.000000,602.000000,1479.000000,1717.000000
mean,1000.500000,0.323500,12.079000,2.42850,0.5050,88.799675,1.470854,3.330819,16.524079,0.269610,...,0.366366,0.343774,0.470147,0.165290,2.794074,0.034215,-0.231558,0.028239,3.082854,0.232854
std,577.494589,0.467929,5.886177,0.95726,0.5001,153.847979,0.683722,2.614791,34.613509,0.169764,...,1.118526,1.140478,2.248409,0.976969,63.726119,0.181922,65.625861,0.165793,11.572512,1.336632
min,1.000000,0.000000,1.000000,1.00000,0.0000,0.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,-100.000000,0.000000,-100.000000,0.000000,0.000000,0.000000
25%,500.750000,0.000000,8.000000,2.00000,0.0000,7.838078,1.000000,1.000000,0.779690,0.137653,...,0.000000,0.000000,0.000000,0.000000,-25.357142,0.000000,-33.333332,0.000000,0.000000,0.000000
50%,1000.500000,0.000000,13.000000,2.00000,1.0000,25.237235,1.000000,3.000000,3.044440,0.223523,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1500.250000,1.000000,17.000000,3.00000,1.0000,80.623119,2.000000,6.000000,13.790535,0.366669,...,0.000000,0.000000,0.091060,0.010243,14.285714,0.000000,18.621060,0.000000,1.502209,0.004328
max,2000.000000,1.000000,21.000000,4.00000,1.0000,991.282959,3.000000,8.000000,302.892059,0.600000,...,8.000000,8.000000,57.938087,17.998640,200.000000,1.000000,200.000000,1.000000,197.691116,38.001801


In [5]:
# Missing values per variable
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=False).head(20)

,Missing,Percent
iap2017x,1398,69.90
iap2017,1398,69.90
umunw,1364,68.20
iap2016,1357,67.85
iap2016x,1357,67.85
iano2017,1137,56.85
iano2016,1032,51.60
qualp,691,34.55
rekp,668,33.40
rek,652,32.60


## 3. Define Key Variables

In [6]:
# Variable labels from metadata
print('Variable Labels:')
for col, label in meta.column_names_to_labels.items():
    if label:
        print(f'{col}: {label}')

Variable Labels:
id: Firm number
ost: East/West for draw
branche: Classification in 21 economic sectors
bran_4: Classification in 4 economic main sectors
filter: Non-innovator
bges: Full-time employees
gk3n: size class
bhsp: Share of graduated employees
bhspno: No graduated employees
um: Total turnover
lp: Labour productivity
lpx: Truncation labour productivity
ex: Exports in 2016
exno: No exports in 2016
umpr1: Sales share of best-selling product
pd: Produktinnovation 2013-2015
umneu: Share improved products/service (2013-2015)
umunw: Share equal products/service (2013-2015)
mneu: Market innovations (2013-2015)
mneup: Share net income market innovations (2013-2015)
novor: Innovation without forerunner products(2013-2015)
novorp: Share of turnover from innovations without forerunner prod-ucts (2013-2015)
pz: Process innovations
rek: Cost reduction
rekp: Cost reduction unit costs
qual: Quality improvement by process innovations
qualp: Increasing turnover as result of quality improvement

### Variable Selection:

**Outcome Variable:** `novor` (Product novelty - binary)
- Measures whether a company introduced a new product
- Suitable for logistic regression (limited dependent variable)

**Main Explanatory Variable:** `digisw1` (Digitalization - software usage)
- Measures the degree of digitalization in the company

**Expected Relationship:** Higher digitalization leads to higher probability of product innovation

**Theoretical Mechanism:** Digital tools enable better information processing, coordination, and innovation capabilities.

In [7]:
# Prepare data for analysis
analysis_df = df.copy()

# Convert object columns to numeric where possible
for col in analysis_df.columns:
    if analysis_df[col].dtype == 'object':
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors='ignore')

# Show distribution of outcome variable
print('Distribution of Outcome Variable (novor):')
print(analysis_df['novor'].value_counts(dropna=False))
print(f'\nProportion with novor=1: {(analysis_df["novor"] == 1).mean():.2%}')

Distribution of Outcome Variable (novor):
novor
0.0    1250
NaN     459
1.0     291
Name: count, dtype: int64

Proportion with novor=1: 14.55%


In [8]:
# Distribution of digitalization variable
print('Distribution of Digitalization Variable (digisw1):')
print(analysis_df['digisw1'].value_counts(dropna=False))

Distribution of Digitalization Variable (digisw1):
digisw1
0.0    710
1.0    526
2.0    381
3.0    202
NaN    181
Name: count, dtype: int64


## 4. Regression Analysis (using pyfixest)

In [9]:
# Prepare dataset for regression
regression_vars = ['novor', 'digisw1', 'branche', 'ost', 'bges', 'lp', 'ex', 'qual']
reg_df = analysis_df[regression_vars].copy()

# Convert to numeric
for col in reg_df.columns:
    reg_df[col] = pd.to_numeric(reg_df[col], errors='coerce')

# Drop missing values
reg_df = reg_df.dropna()
print(f'Observations for regression: {len(reg_df)}')

# Convert branche to categorical for fixed effects
reg_df['branche'] = reg_df['branche'].astype('category')

Observations for regression: 976


In [10]:
# Model 1: Baseline model (only main variable)
model1 = pf.feols('novor ~ digisw1', data=reg_df)

# Model 2: With control variables (demographics)
model2 = pf.feols('novor ~ digisw1 + ost + bges + lp', data=reg_df)

# Model 3: With additional control variables
model3 = pf.feols('novor ~ digisw1 + ost + bges + lp + ex + qual', data=reg_df)

# Model 4: With industry fixed effects
model4 = pf.feols('novor ~ digisw1 + ost + bges + lp + ex + qual | branche', data=reg_df)

# Display model 4 results
model4.tidy()

,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
digisw1,-0.004043,0.007660,-0.527839,0.597735,-0.019076,0.010990
ost,0.050032,0.017052,2.934133,0.003425,0.016569,0.083496
bges,0.000286,0.000076,3.781209,0.000166,0.000137,0.000434
lp,0.027211,0.053309,0.510447,0.609857,-0.077405,0.131828
ex,0.000068,0.000552,0.122666,0.902397,-0.001015,0.001151
qual,0.572724,0.024902,22.999232,0.000000,0.523855,0.621594


In [11]:
# Create regression table using pyfixest
etable_df = pf.etable([model1, model2, model3, model4], 
                  model_heads=['Model 1', 'Model 2', 'Model 3', 'Model 4'],
                  stars=True,
                  type='df')
etable_df

,est1,est2,est3,est4
depvar,novor,novor,novor,novor
digisw1,0.009 \n (0.010),0.005 \n (0.010),-0.005 \n (0.008),-0.004 \n (0.008)
ost,,0.031 \n (0.022),0.044** \n (0.017),0.050** \n (0.017)
bges,,0.001*** \n (0.000),0.000*** \n (0.000),0.000*** \n (0.000)
lp,,0.026 \n (0.061),0.010 \n (0.048),0.027 \n (0.053)
ex,,,0.000 \n (0.001),0.000 \n (0.001)
qual,,,0.591*** \n (0.023),0.573*** \n (0.025)
Intercept,0.116*** \n (0.014),0.055* \n (0.023),0.006 \n (0.018),
branche,-,-,-,x
Observations,976,976,976,976


## 5. Inference with Robust Standard Errors

In [12]:
# Model 4 with heteroskedasticity-robust standard errors
# Note: HC1 is used for fixed effects models (HC2/HC3 not supported)
model4_robust = pf.feols('novor ~ digisw1 + ost + bges + lp + ex + qual | branche', 
                         data=reg_df, vcov='hetero')

# Compare standard errors
print('=== Comparison: Non-robust vs. Robust Standard Errors ===\n')

comparison_df = pd.DataFrame({
    'Non-robust SE': model4.se(),
    'Robust SE (HC1)': model4_robust.se(),
    'Non-robust P-value': model4.pvalue(),
    'Robust P-value': model4_robust.pvalue()
})

comparison_df['SE Diff (%)'] = ((comparison_df['Robust SE (HC1)'] - comparison_df['Non-robust SE']) / 
                                 comparison_df['Non-robust SE'] * 100).round(2)

print(comparison_df.round(4))

=== Comparison: Non-robust vs. Robust Standard Errors ===

             Non-robust SE  Robust SE (HC1)  Non-robust P-value  \
Coefficient                                                       
digisw1             0.0077           0.0073              0.5977   
ost                 0.0171           0.0166              0.0034   
bges                0.0001           0.0001              0.0002   
lp                  0.0533           0.0495              0.6099   
ex                  0.0006           0.0007              0.9024   
qual                0.0249           0.0444              0.0000   

             Robust P-value  SE Diff (%)  
Coefficient                               
digisw1              0.5787        -4.99  
ost                  0.0026        -2.88  
bges                 0.0055        35.85  
lp                   0.5826        -7.16  
ex                   0.9273        34.43  
qual                 0.0000        78.32  


## 6. Interaction Effects

In [13]:
# Interaction term: Digitalization x Firm Size
# Hypothesis: The effect of digitalization on product innovation 
# is stronger in larger firms

# Model with interaction using formula syntax
model_interaction = pf.feols('novor ~ digisw1 * bges + ost + lp + ex + qual | branche', 
                             data=reg_df)

print('Model with interaction term estimated')
print('\n=== Coefficients ===')
model_interaction.tidy()

Model with interaction term estimated

=== Coefficients ===


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
digisw1,-0.007318,0.008653,-0.845674,0.397948,-0.024300,0.009664
bges,0.000232,0.000100,2.320211,0.020541,0.000036,0.000429
ost,0.050156,0.017056,2.940755,0.003354,0.016685,0.083627
lp,0.023216,0.053544,0.433586,0.664688,-0.081862,0.128293
ex,0.000071,0.000552,0.128785,0.897555,-0.001012,0.001154
qual,0.572437,0.024909,22.981321,0.000000,0.523555,0.621320
digisw1:bges,0.000051,0.000063,0.814042,0.415825,-0.000072,0.000174


## 7. Visualization

The logistic regression plot visualizes the relationship between digitalization and innovation probability.

In [14]:
# Use marginaleffects to compute average slopes
avg_slp = avg_slopes(model_interaction, variables='digisw1', by='bges')
print('Average Slopes of Digitalization by Firm Size:')
print(avg_slp)

Average Slopes of Digitalization by Firm Size:
shape: (976, 8)
┌──────────┬──────────┬───────────┬────────┬─────────┬──────┬─────────┬─────────┐
│ bges     ┆ Estimate ┆ Std.Error ┆ z      ┆ P(>|z|) ┆ S    ┆ 2.5%    ┆ 97.5%   │
│ ---      ┆ ---      ┆ ---       ┆ ---    ┆ ---     ┆ ---  ┆ ---     ┆ ---     │
│ str      ┆ str      ┆ str       ┆ str    ┆ str     ┆ str  ┆ str     ┆ str     │
╞══════════╪══════════╪═══════════╪════════╪═════════╪══════╪═════════╪═════════╡
│ 7.88e-07 ┆ -0.00732 ┆ 0.00866   ┆ -0.845 ┆ 0.398   ┆ 1.33 ┆ -0.0243 ┆ 0.00965 │
│ 1.01e-06 ┆ -0.00732 ┆ 0.00865   ┆ -0.846 ┆ 0.398   ┆ 1.33 ┆ -0.0243 ┆ 0.00964 │
│ 0.116    ┆ -0.00731 ┆ 0.00865   ┆ -0.846 ┆ 0.398   ┆ 1.33 ┆ -0.0243 ┆ 0.00964 │
│ 0.153    ┆ -0.00731 ┆ 0.00865   ┆ -0.845 ┆ 0.398   ┆ 1.33 ┆ -0.0243 ┆ 0.00964 │
│ 0.45     ┆ -0.0073  ┆ 0.00864   ┆ -0.844 ┆ 0.399   ┆ 1.33 ┆ -0.0242 ┆ 0.00964 │
│ …        ┆ …        ┆ …         ┆ …      ┆ …       ┆ …    ┆ …       ┆ …       │
│ 769      ┆ 0.0319   ┆ 0.0448    ┆

In [15]:
# Logistic Regression Plot: Digitalization vs Innovation Probability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Raw data with logistic fit
ax = axes[0]

# Create jittered data for visualization
np.random.seed(42)
jitter = np.random.normal(0, 0.15, len(reg_df))
digisw1_jittered = reg_df['digisw1'] + jitter

# Plot raw data
ax.scatter(digisw1_jittered[reg_df['novor'] == 0], 
           reg_df.loc[reg_df['novor'] == 0, 'novor'], 
           alpha=0.3, label='No Innovation', color='gray', s=30)
ax.scatter(digisw1_jittered[reg_df['novor'] == 1], 
           reg_df.loc[reg_df['novor'] == 1, 'novor'], 
           alpha=0.5, label='Innovation', color='steelblue', s=40)

# Plot logistic curve using tidy() output
tidy_df = model4.tidy()
beta_digisw1 = tidy_df.loc['digisw1', 'Estimate']
beta_intercept = 0.006  # From Model 4 intercept

x_vals = np.linspace(1, 5, 100)
logit_vals = beta_digisw1 * x_vals + beta_intercept
prob_vals = 1 / (1 + np.exp(-logit_vals))
ax.plot(x_vals, prob_vals, 'r-', linewidth=2.5, label='Logistic Fit')

ax.set_xlabel('Digitalization (digisw1)', fontsize=11)
ax.set_ylabel('Probability of Product Innovation', fontsize=11)
ax.set_title('Panel A: Logistic Regression Fit', fontsize=12, fontweight='bold')
ax.legend(loc='upper left')
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# Panel B: Predicted probabilities with confidence intervals
ax = axes[1]

# Get predictions using marginaleffects
pred = predictions(model4, variables='digisw1')
pred_pd = pred.to_pandas()

# Group by digisw1 and compute means and CIs
grouped = pred_pd.groupby('digisw1')['estimate'].agg(['mean', 'std', 'count'])
grouped['se'] = grouped['std'] / np.sqrt(grouped['count'])
grouped['ci_lower'] = grouped['mean'] - 1.96 * grouped['se']
grouped['ci_upper'] = grouped['mean'] + 1.96 * grouped['se']

# Plot predicted probabilities
ax.plot(grouped.index, grouped['mean'], 'bo-', linewidth=2, markersize=8, label='Predicted Probability')
ax.fill_between(grouped.index, grouped['ci_lower'], grouped['ci_upper'], 
                alpha=0.3, color='blue', label='95% CI')

ax.set_xlabel('Digitalization (digisw1)', fontsize=11)
ax.set_ylabel('Predicted Probability of Innovation', fontsize=11)
ax.set_title('Panel B: Predicted Probabilities with 95% CI', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/logistic_regression_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print('Logistic regression plot saved: data/logistic_regression_plot.png')

Logistic regression plot saved: data/logistic_regression_plot.png


## 8. Formatting Regression Table for Word/LaTeX

In [16]:
# Export regression table as LaTeX using pyfixest
latex_table = pf.etable([model1, model2, model3, model4], 
                    model_heads=['Model 1', 'Model 2', 'Model 3', 'Model 4'],
                    type='tex',
                    stars=True)

with open('data/regression_table.tex', 'w') as f:
    f.write(latex_table)
print('LaTeX table saved: data/regression_table.tex')
print('\nLaTeX Table Preview:')
print(latex_table[:500])

LaTeX table saved: data/regression_table.tex

LaTeX Table Preview:
\renewcommand\cellalign{t}
\begin{threeparttable}
\begin{tabular}{lccccc}
\toprule
 & \multicolumn{4}{c}{novor} \\
\cmidrule(lr){2-5} 
 & Model 1 & Model 2 & Model 3 & Model 4 \\
 & (1) & (2) & (3) & (4) \\
\midrule
\addlinespace
digisw1 & \makecell{0.009 \\ (0.010)} & \makecell{0.005 \\ (0.010)} & \makecell{-0.005 \\ (0.008)} & \makecell{-0.004 \\ (0.008)} \\
ost &  & \makecell{0.031 \\ (0.022)} & \makecell{0.044** \\ (0.017)} & \makecell{0.050** \\ (0.017)} \\
bges &  & \makecell{0.001*** \\ (


In [17]:
# Create formatted table as DataFrame
table_df = pf.etable([model1, model2, model3, model4], 
                 model_heads=['Model 1', 'Model 2', 'Model 3', 'Model 4'],
                 type='df',
                 stars=True)

table_df.to_csv('data/regression_table.csv', index=False)

print('\nFormatted Regression Table:')
print(table_df.to_string(index=False))
print('\nTable saved: data/regression_table.csv')


Formatted Regression Table:
               est1                est2                est3                est4
              novor               novor               novor               novor
   0.009 \n (0.010)    0.005 \n (0.010)   -0.005 \n (0.008)   -0.004 \n (0.008)
                       0.031 \n (0.022)  0.044** \n (0.017)  0.050** \n (0.017)
                    0.001*** \n (0.000) 0.000*** \n (0.000) 0.000*** \n (0.000)
                       0.026 \n (0.061)    0.010 \n (0.048)    0.027 \n (0.053)
                                           0.000 \n (0.001)    0.000 \n (0.001)
                                        0.591*** \n (0.023) 0.573*** \n (0.025)
0.116*** \n (0.014)   0.055* \n (0.023)    0.006 \n (0.018)                    
                  -                   -                   -                   x
                976                 976                 976                 976
                iid                 iid                 iid                 iid
           

## 9. Summary of Results

In [18]:
print('=' * 70)
print('ANALYSIS SUMMARY')
print('=' * 70)

print('\n1. DATA:')
print(f'   - Observations: {len(reg_df)}')
print(f'   - Proportion innovative firms (novor=1): {(reg_df["novor"] == 1).mean():.2%}')

print('\n2. MAIN RESULT (Model 4 with Industry Fixed Effects):')
tidy_df = model4.tidy()
if 'digisw1' in tidy_df.index:
    digisw_coef = tidy_df.loc['digisw1', 'Estimate']
    digisw_pval = tidy_df.loc['digisw1', 'Pr(>|t|)']
    digisw_se = tidy_df.loc['digisw1', 'Std. Error']
    print(f'   - Coefficient Digitalization: {digisw_coef:.4f}')
    print(f'   - Standard Error: {digisw_se:.4f}')
    print(f'   - p-value: {digisw_pval:.4f}')
else:
    print('   - digisw1 not in model')

print('\n3. ROBUSTNESS:')
print(f'   - Standard Error (non-robust): {model4.se().get("digisw1", "N/A")}')
print(f'   - Standard Error (robust HC1): {model4_robust.se().get("digisw1", "N/A")}')

print('\n4. INTERACTION EFFECT:')
print(f'   - Interaction Digitalization x Firm Size analyzed')
print(f'   - See plot: data/marginal_effects_interaction.png')

print('\n' + '=' * 70)

ANALYSIS SUMMARY

1. DATA:
   - Observations: 976
   - Proportion innovative firms (novor=1): 12.40%

2. MAIN RESULT (Model 4 with Industry Fixed Effects):
   - Coefficient Digitalization: -0.0040
   - Standard Error: 0.0077
   - p-value: 0.5977

3. ROBUSTNESS:
   - Standard Error (non-robust): 0.007660249910400691
   - Standard Error (robust HC1): 0.007278342435165764

4. INTERACTION EFFECT:
   - Interaction Digitalization x Firm Size analyzed
   - See plot: data/marginal_effects_interaction.png

